## Food Delivery Time Prediction using Logistic Regression

### Importing required Libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

### Loading the Dataset

In [2]:
data = pd.read_csv(r"C:\Users\WELCOME\Downloads\Food Delivery Excel file\Food_Delivery_Times.csv")
data

,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37
4,412,19.03,Clear,Low,Morning,Bike,16,5.0,68
...,...,...,...,...,...,...,...,...,...
995,107,8.50,Clear,High,Evening,Car,13,3.0,54
996,271,16.28,Rainy,Low,Morning,Scooter,8,9.0,71
997,861,15.62,Snowy,High,Evening,Scooter,26,2.0,81
998,436,14.17,Clear,Low,Afternoon,Bike,8,0.0,55


### Data Understanding

In [3]:
data['Delivery_Time_min'].unique()


array([ 43,  84,  59,  37,  68,  57,  49,  46,  35,  73,  88,  76,  53,
        36,  33,  50,  24,  27,  47,  72,  58,  56,  64,  70, 123,  52,
       108,  45, 111,  44,  61,  34,  67,  32,   8, 104,  31,  23,  82,
        69,  60,  40,  38,  54,  87,  62,  22,  42,  51,  41,  48,  92,
        71,  65,  28,  74,  14,  30,  94,  80,  91,  79,  77,  26, 141,
        78,  29, 105,  16, 116,  75,  25, 113,  17,  66,  85, 100,  21,
        89,  55,  96,  13,  90, 112,  63,  39, 101,  93,  83, 109,  20,
        19, 153, 102, 103,  86, 115,  15,  98, 106,  97,  18,  99,  81,
        95, 126, 122, 114])

In [4]:
data.shape


(1000, 9)

In [5]:
data.columns


Index(['Order_ID', 'Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
       'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs',
       'Delivery_Time_min'],
      dtype='object')

In [6]:
data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Order_ID                1000 non-null   int64  
 1   Distance_km             1000 non-null   float64
 2   Weather                 970 non-null    object 
 3   Traffic_Level           970 non-null    object 
 4   Time_of_Day             970 non-null    object 
 5   Vehicle_Type            1000 non-null   object 
 6   Preparation_Time_min    1000 non-null   int64  
 7   Courier_Experience_yrs  970 non-null    float64
 8   Delivery_Time_min       1000 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 70.4+ KB


In [7]:
data.head()


,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37
4,412,19.03,Clear,Low,Morning,Bike,16,5.0,68


In [8]:
data.tail()


,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
995,107,8.50,Clear,High,Evening,Car,13,3.0,54
996,271,16.28,Rainy,Low,Morning,Scooter,8,9.0,71
997,861,15.62,Snowy,High,Evening,Scooter,26,2.0,81
998,436,14.17,Clear,Low,Afternoon,Bike,8,0.0,55
999,103,6.63,Foggy,Low,Night,Scooter,24,3.0,58


In [9]:
data.isnull().sum()


Order_ID                   0
Distance_km                0
Weather                   30
Traffic_Level             30
Time_of_Day               30
Vehicle_Type               0
Preparation_Time_min       0
Courier_Experience_yrs    30
Delivery_Time_min          0
dtype: int64

### Data cleaning

In [10]:
data = data.drop(["Order_ID"], axis=1)


In [11]:
### Filling Null values
data["Weather"] = data["Weather"].fillna("Unknown")
data["Traffic_Level"] = data["Traffic_Level"].fillna("Unknown")
data["Time_of_Day"] = data["Time_of_Day"].fillna("Unknown")
data["Courier_Experience_yrs"] = data["Courier_Experience_yrs"].fillna(data["Courier_Experience_yrs"].mean())


In [12]:
data.isnull().sum().sum()


np.int64(0)

In [13]:
data.dtypes


Distance_km               float64
Weather                    object
Traffic_Level              object
Time_of_Day                object
Vehicle_Type               object
Preparation_Time_min        int64
Courier_Experience_yrs    float64
Delivery_Time_min           int64
dtype: object

### Encoding categorical data

In [14]:
data = pd.get_dummies(data,columns=['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type'], drop_first=True)
data.replace({True:1, False:0}, inplace=True)
data["Delivery_Time_min"] = data["Delivery_Time_min"].map(lambda x : 1 if x<30 else 0)


In [15]:
data["Delivery_Time_min"].value_counts()

Delivery_Time_min
0    897
1    103
Name: count, dtype: int64

In [16]:
data.head()

,Distance_km,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min,Weather_Foggy,Weather_Rainy,Weather_Snowy,Weather_Unknown,Weather_Windy,Traffic_Level_Low,Traffic_Level_Medium,Traffic_Level_Unknown,Time_of_Day_Evening,Time_of_Day_Morning,Time_of_Day_Night,Time_of_Day_Unknown,Vehicle_Type_Car,Vehicle_Type_Scooter
0,7.93,12,1.0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1
1,16.42,20,2.0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0
2,9.52,28,1.0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,1
3,7.44,5,1.0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1
4,19.03,16,5.0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0


### Feature and Target selection

In [17]:
feature = data.drop(columns='Delivery_Time_min', axis=1)
target = data['Delivery_Time_min']
print("\n Features :\n",feature)
print("\n Target :\n",target)



 Features :
      Distance_km  Preparation_Time_min  Courier_Experience_yrs  Weather_Foggy  \
0           7.93                    12                     1.0              0   
1          16.42                    20                     2.0              0   
2           9.52                    28                     1.0              1   
3           7.44                     5                     1.0              0   
4          19.03                    16                     5.0              0   
..           ...                   ...                     ...            ...   
995         8.50                    13                     3.0              0   
996        16.28                     8                     9.0              0   
997        15.62                    26                     2.0              0   
998        14.17                     8                     0.0              0   
999         6.63                    24                     3.0              1   

     Weather_

### Feature Scaling

In [18]:
scaler = StandardScaler()
scaler.fit(feature)

standardized_data = scaler.transform(feature)
print(standardized_data)

feature = standardized_data
target = data['Delivery_Time_min']

print("\n Features :\n",feature)
print("\n Target :\n",target)


[[-0.37408542 -0.69185319 -1.24766455 ... -0.17586311 -0.49217479
   1.5202823 ]
 [ 1.11700846  0.41911139 -0.89909468 ... -0.17586311 -0.49217479
  -0.65777257]
 [-0.09483462  1.53007597 -1.24766455 ... -0.17586311 -0.49217479
   1.5202823 ]
 ...
 [ 0.97650491  1.25233482 -0.89909468 ... -0.17586311 -0.49217479
   1.5202823 ]
 [ 0.72184224 -1.24733548 -1.59623443 ... -0.17586311 -0.49217479
  -0.65777257]
 [-0.60240368  0.97459368 -0.5505248  ... -0.17586311 -0.49217479
   1.5202823 ]]

 Features :
 [[-0.37408542 -0.69185319 -1.24766455 ... -0.17586311 -0.49217479
   1.5202823 ]
 [ 1.11700846  0.41911139 -0.89909468 ... -0.17586311 -0.49217479
  -0.65777257]
 [-0.09483462  1.53007597 -1.24766455 ... -0.17586311 -0.49217479
   1.5202823 ]
 ...
 [ 0.97650491  1.25233482 -0.89909468 ... -0.17586311 -0.49217479
   1.5202823 ]
 [ 0.72184224 -1.24733548 -1.59623443 ... -0.17586311 -0.49217479
  -0.65777257]
 [-0.60240368  0.97459368 -0.5505248  ... -0.17586311 -0.49217479
   1.5202823 ]]

 

### Data Splitting

In [19]:
x_train, x_test, y_train, y_test = train_test_split(feature,target,test_size=0.2,random_state=42)
print(feature.shape, x_train.shape, x_test.shape)


(1000, 17) (800, 17) (200, 17)


### Model Evaluation

In [20]:
classifier = LogisticRegression(class_weight='balanced')
classifier.fit(x_train, y_train)


LogisticRegression(class_weight='balanced')

### Predictions

In [21]:
# Prediction on training data
y_prediction = classifier.predict(x_train)
training_data_accuracy = accuracy_score(y_train, y_prediction)
print("Accuracy score of training data :",training_data_accuracy) 

# Prediction on testing data
y_prediction = classifier.predict(x_test)
testing_data_accuracy = accuracy_score(y_test, y_prediction)
print("Accuracy score of testing data :",testing_data_accuracy)


Accuracy score of training data : 0.94875
Accuracy score of testing data : 0.92


In [22]:
data.head()


,Distance_km,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min,Weather_Foggy,Weather_Rainy,Weather_Snowy,Weather_Unknown,Weather_Windy,Traffic_Level_Low,Traffic_Level_Medium,Traffic_Level_Unknown,Time_of_Day_Evening,Time_of_Day_Morning,Time_of_Day_Night,Time_of_Day_Unknown,Vehicle_Type_Car,Vehicle_Type_Scooter
0,7.93,12,1.0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1
1,16.42,20,2.0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0
2,9.52,28,1.0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,1
3,7.44,5,1.0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1
4,19.03,16,5.0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0


### Resulting Predictions

#### Result 1 :

In [23]:
input_data = (25, 30, 10, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1)
input_numpy_array = np.asarray(input_data)
input_data_reshape = input_numpy_array.reshape(1,-1)

std_data = scaler.transform(input_data_reshape)
print(std_data)

prediction = classifier.predict(std_data)
prediction_proba = classifier.predict_proba(std_data)

print(prediction)
print("\n *** FOOD DELIVERY PREDICTION ***")

if prediction[0] == 1:
    print("\n Fast Delivery")
else:
    print("\n Slow Delivery")
    

[[ 2.62390899  1.80781711  1.88946435 -0.33886163 -0.50624244 -0.32774947
   5.6862407  -0.32587527  1.26923838 -0.79959006 -0.17586311 -0.64376017
  -0.66714819  3.28096112 -0.17586311 -0.49217479  1.5202823 ]]
[0]

 *** FOOD DELIVERY PREDICTION ***

 Slow Delivery


#### Result 2 :

In [24]:
input_data = (10, 10, 10, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1)
input_numpy_array = np.asarray(input_data)
input_data_reshape = input_numpy_array.reshape(1,-1)

std_data = scaler.transform(input_data_reshape)
print(std_data)

prediction = classifier.predict(std_data)
prediction_proba = classifier.predict_proba(std_data)

print(prediction)
print("\n *** FOOD DELIVERY PREDICTION ***")

if prediction[0] == 1:
    print("\n Fast Delivery")
else:
    print("\n Slow Delivery")
    

[[-0.0105325  -0.96959434  1.88946435 -0.33886163 -0.50624244 -0.32774947
   5.6862407  -0.32587527  1.26923838 -0.79959006 -0.17586311 -0.64376017
  -0.66714819  3.28096112 -0.17586311 -0.49217479  1.5202823 ]]
[1]

 *** FOOD DELIVERY PREDICTION ***

 Fast Delivery


### Overall Insights
- The model effectively predicts whether a delivery will be fast (<30 min) or slow (≥ 30 min) using features like distance, preparation time, courier experience, traffic, weather, and time of day.
- Fast deliveries are associated with short distances, low preparation time, experienced couriers, favorable weather, and light traffic.
- Slow deliveries occur due to long distances, high preparation times, less experienced couriers, adverse weather, or peak traffic periods.
- Probability scores from the model provide confidence levels, helping identify orders likely to be slow for proactive action.
- Key features such as distance, preparation time, and traffic conditions have the most significant impact on delivery speed, guiding optimization strategies.